# PRIORITEST-STAR — Fusion guidée par similarité et contraste inter-vues

**TP de démarrage, données PROMISE Ant réelles.** Téléversez l’archive `PRIORITEST_STAR_DOSSIER_ETUDIANTS.zip` dans Colab. Le premier entraînement s’exécute sur CPU ou GPU si disponible. Le protocole évalue les défauts de la version cible à partir de ses métriques ; il ne prouve **pas** une prévision des défauts de la version suivante.

In [ ]:
from google.colab import files
uploaded = files.upload()
import zipfile, pathlib
archive = next((k for k in uploaded if k.endswith(".zip")), None)
assert archive is not None, "Veuillez sélectionner PRIORITEST_STAR_DOSSIER_ETUDIANTS.zip"
with zipfile.ZipFile(archive) as z: z.extractall("/content")
ROOT = pathlib.Path("/content/PRIORITEST_STAR_CrossView")
assert (ROOT/"src/train.py").exists()
print("Kit installé :", ROOT)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(ROOT/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=ROOT,check=True)

## Quatre ablations : fusion classique, fusion par similarité, contraste inter-vues, architecture complète

Sélection sur Ant 1.6, test final sur Ant 1.7. La graine peut être modifiée pour tester la stabilité.

In [ ]:
import torch
device="cuda" if torch.cuda.is_available() else "cpu"
print("Appareil disponible :",device)
subprocess.run([sys.executable,"src/train.py","--data","data","--out","results/colab_seed42","--epochs","35","--seed","42","--device",device],cwd=ROOT,check=True)

In [ ]:
import json,pandas as pd
res=json.loads((ROOT/"results/colab_seed42/metrics.json").read_text())
pd.DataFrame([{ "variante":name,"AP_validation":v["1.6"]["AP"],"AP_test":v["1.7"]["AP"],"Recall_20pct_LOC":v["1.7"]["Recall@20pct_LOC"]} for name,v in res["models"].items()])

## Travail étudiant

1. Interpréter le coefficient d’accord cosinus entre la vue temporelle et structurelle.
2. Comparer les quatre variantes avec trois graines et leurs écarts.
3. Tester l’impact de la température InfoNCE et du poids contrastif (sélection sur validation uniquement).
4. Comparer les budgets LOC ; expliquer pourquoi une bonne AP ne garantit pas un meilleur rappel sous budget.
5. Proposer un traitement des *false negatives* InfoNCE entre classes défectueuses similaires.

**Limites :** source Ant seulement ; graphes d’imports partiels ; aucun essai Colab GPU certifié dans ce kit ; expérience non prospective.